### Assignment 2: Tracking a Panda in a Bamboo Grove: Exponential Smoothing vs. Running Mean

##### Group 6: Pavel Kuznetsov, Olga Tiupina, Sofya Konstantinova, Skoltech, 2025

First of all we created function collection for each milestone in the 2 assigment:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_true_trajectory(num, sigma, initial_condition):
    """
    Generate a panda walking trajectory using random walk model
    """
    X = np.zeros(num)
    X[0] = initial_condition
    
    # normally distributed random noise with a mean of zero and a variance of 𝜎𝑤^2
    w = np.random.normal(0, np.sqrt(sigma), num)
    
    # the random walk model
    for i in range(1, num):
        X[i] = X[i-1] + w[i-1]
    
    return X, w

def generate_measurements(true_trajectory, sigma):
    """
    Generate a panda walking measurements using random walk model
    """

    num = len(true_trajectory)
    z = np.zeros(num)
    
    n = np.random.normal(0, np.sqrt(sigma), num)    
    
    for i in range(num):
        z[i] = true_trajectory[i] + n[i]
    
    return z, n

def generate_exponential_smoothing(measurements, alfa):
    """
    Generate Exponential smoothing
    """
    num = len(measurements)
    X_ex = np.zeros(num)
    X_ex[0] = measurements[0]
    
    # exponential smoothing
    for i in range(1, num):
        X_ex[i] = alfa * measurements[i] + (1 - alfa) * X_ex[i-1] 
    
    return X_ex


def calculate_optimal_smoothing(sigma_w, sigma_n):
    """
    Calculate optimal smoothing
    """
   
    chi = sigma_w / sigma_n
    alpha = (-chi + np.sqrt(chi ** 2 + 4 * chi)) / 2
    return alpha

def math_expectation(N, value, first_element):
    """
    Find mean 
    """
    expectation = 0.0
    for i in range(first_element, N):
        expectation += value[i]**2
    expectation = expectation / (N - first_element)
    return expectation


def real_sigma_search(N, w, n):  
    """
    Find real sigma
    """  
    
    v = np.zeros(N)
    p = np.zeros(N)

    for i in range(1, N):
        v[i] = w[i] + n[i] - n[i-1]

    for i in range(2, N):
        p[i] = w[i] + w[i-1] + n[i] - n[i-2]

    mean_v = math_expectation(N, v, 1)
    mean_p = math_expectation(N, p, 2)

    sigma_w2 = mean_p - mean_v
    sigma_n2 = 0.5*(2*mean_v - mean_p)

    print(f"value for {N} elements:")
    print(f"sigma_w2 = {sigma_w2}")
    print(f"sigma_n2 = {sigma_n2}")
    return sigma_w2, sigma_n2


def M_determination(alpha):
    """
    Find window size M
    """
    M = (2.0 - alpha) / alpha
    return int(round(M))


def running_mean(alpha, measurements):
    """
    Running mean б
    """

    M = M_determination(alpha)
    print(f"M = {M}")

    n = len(measurements)
    x = np.zeros(n)  
    half = (M - 1) // 2  
    
    #leftpart
    for i in range(half):
        x[i] = np.mean(measurements[:i + half + 1])
    
    #centralpart
    for i in range(half, n - half):
        window_sum = 0
        for k in range(i - half, i + half + 1):
            window_sum += measurements[k]
        x[i] = window_sum / M
    
    #rightpart
    for i in range(n - half, n):
        x[i] = np.mean(measurements[i - half:])
    
    return x

In [ ]:
fontsize_title = 25
fontsize_label = 20
fontsize_leg = 18

In the first part we built panda random calm walking model using random noise w with given covariance and zero mean. After this we created measurements for our model to simulate real world condition (random noise n with given covariance and zero mean). We did it for 300 and 3000 points. Also we calculated estimated sigmas for noise of process and noise of measurments. 

In [ ]:
# Parameters
sigma_w1 = 13  
initial_condition = 10
sigma_n1 = 8

# Generate trajectories and measurements
calm_true_trajectory_3000, w_3000 = generate_true_trajectory(num=3000, sigma=sigma_w1, initial_condition=initial_condition)
calm_true_trajectory_300, w_300 = generate_true_trajectory(num=300, sigma=sigma_w1, initial_condition=initial_condition)

calm_measurements_3000, n_3000 = generate_measurements(true_trajectory=calm_true_trajectory_3000, sigma=sigma_n1)
calm_measurements_300, n_300 = generate_measurements(true_trajectory=calm_true_trajectory_300, sigma=sigma_n1)

N_300 = 300
N_3000 = 3000
sigma_w_estimated_300, sigma_p_estimated_300 = real_sigma_search(N_300, w_300, n_300)
sigma_w_estimated_3000, sigma_p_estimated_3000 = real_sigma_search(N_3000, w_3000, n_3000)

# absolute error
abs_error_sigma_w2_300 = abs(sigma_w1 - sigma_w_estimated_300)
abs_error_sigma_w2_3000 = abs(sigma_w1 - sigma_w_estimated_3000)
abs_error_sigma_p2_300 = abs(sigma_n1 - sigma_p_estimated_300)
abs_error_sigma_p2_3000 = abs(sigma_n1 - sigma_p_estimated_3000)

print(f"abs_error_sigma_w2_300 = {abs_error_sigma_w2_300}")
print(f"abs_error_sigma_w2_3000 = {abs_error_sigma_w2_3000}")
print(f"abs_error_sigma_p2_300 = {abs_error_sigma_p2_300}")
print(f"abs_error_sigma_p2_3000 = {abs_error_sigma_p2_3000}")

Comparing true values and estimated sigmas shows low difference between calculated and given values. It shows that our model is correct. For bigger amount of points we get better accurancy. So 3000 point give values which are near to the true sigmas.

We found optimal smoothing coefficients for 300 points and 3000 points:

In [ ]:
alfa1_300 = calculate_optimal_smoothing(sigma_w_estimated_300, sigma_p_estimated_300)
alfa1_3000 = calculate_optimal_smoothing(sigma_w_estimated_3000, sigma_p_estimated_3000)
print(f"Optimal smoothing coefficient for 300 points: {alfa1_300:.4f}")
print(f"Optimal smoothing coefficient for 3000 points: {alfa1_3000:.4f}")

# Generate exponential smoothing
calm_ex_smoothing_3000 = generate_exponential_smoothing(calm_measurements_3000, alfa=alfa1_3000)
calm_ex_smoothing_300 = generate_exponential_smoothing(calm_measurements_300, alfa=alfa1_300)

We ploted all trajectories: Calm Walk: Measurements, True Trajectory and Exponential Smoothing for 3000 points and for 300 points:

In [ ]:
plt.figure(figsize=(25,16))

# Plot for 3000 points
plt.subplot(2, 1, 1)

plt.plot(calm_true_trajectory_3000,linestyle='--',linewidth=1, color='black', alpha=1, label='True trajectory')
plt.plot(calm_measurements_3000, linewidth=1, color='blue', alpha=0.6, label='Measurements')
plt.plot(calm_ex_smoothing_3000, linewidth=1, color='red', alpha=1, label='Exponential smoothing')
plt.title('\n Fig.1. Calm Walk: Measurements, True Trajectory and Exponential Smoothing - 3000 points \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(-5, 3005)
plt.grid(True, alpha=0.3)

plt.legend(fontsize=fontsize_leg)

# Plot for 3000 points zoom between 150 and 350 numbers
plt.subplot(2, 1, 2)

plt.plot(calm_true_trajectory_3000,linestyle='--',linewidth=1, color='black', alpha=1, label='True trajectory')
plt.plot(calm_measurements_3000, linewidth=1, color='blue', alpha=0.6, label='Measurements')
plt.plot(calm_ex_smoothing_3000, linewidth=2, color='red', alpha=1, label='Exponential smoothing')
plt.title('\n Fig.2. Calm Walk: Measurements, True Trajectory and Exponential Smoothing - 3000 points with zooming \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(1250, 1400)
plt.ylim(-90, -10)
plt.grid(True, alpha=0.3)

plt.legend(fontsize=fontsize_leg)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for 300 points
plt.figure(figsize=(25, 8))
					 
plt.plot(calm_true_trajectory_300,linestyle='--',linewidth=1, color='black', alpha=1, label='True trajectory')
plt.plot(calm_measurements_300, linewidth=1, color='blue', alpha=0.6, label='Measurements')
plt.plot(calm_ex_smoothing_300, linewidth=1, color='red', alpha=1, label='Exponential smoothing')
plt.title('\n Fig.3. Calm Walk: Measurements, True Trajectory and Exponential Smoothing - 300 points \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(-5, 305)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

Our estimation is close to true trajectory that shows effectiveness of our smoothing.

For the second part we build walking model in condition of shaky camera that increased noises. 

In [ ]:
#Part II – Shaky Cameras: Testing Under Harsh Conditions
sigma_w2 = 28 ** 2
sigma_n2 = 97 ** 2

alfa2 = calculate_optimal_smoothing(sigma_w2, sigma_n2)
print(f"Optimal smoothing coefficient: {alfa2:.4f}")

# Shaky Cameras trajectory 300 points  
shaky_true_trajectory_300, w_300 = generate_true_trajectory(num=300, sigma=sigma_w2, initial_condition=initial_condition)
# Shaky Cameras measurements 300 points
shaky_measurements_300, n_300 = generate_measurements(true_trajectory=shaky_true_trajectory_300, sigma=sigma_n2)
# Shaky Cameras exponential smoothing
shaky_ex_smoothing_300 = generate_exponential_smoothing(shaky_measurements_300, alfa=alfa2)

x_mean = running_mean(alfa2, shaky_measurements_300)

We found optimal smoothing coefficient that is less than the same coefficient in previouse part due to bigger noise in our measurements. For more effectiveness noise filtration we should use smaller alpha. Also we determined smoothing window size for mean running using the determined alpha, which is equal to 7.

In [ ]:
# Create combined plots similar to the example
plt.figure(figsize=(25, 18))

# Plot for 300 points
plt.subplot(2, 1, 1)
plt.plot(shaky_true_trajectory_300, linestyle='--', color="black", linewidth=1, alpha=1, label='True trajectory')
plt.plot(shaky_measurements_300, color='blue', linewidth=1, alpha=0.6, label='Measurements')
plt.plot(shaky_ex_smoothing_300, color='red', linewidth=1, alpha=1, label='Exponential smoothing')
plt.plot(x_mean, color='green', linewidth=1, alpha=1, label='Running mean')
plt.title('\n Fig.4. Calm Walk: Measurements, True Trajectory, Exponential Smoothing and Running mean - 300 points \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(-5, 305)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

# Plot for 300 points
plt.subplot(2, 1, 2)
plt.plot(shaky_true_trajectory_300, linestyle='--', color="black", linewidth=1, alpha=1, label='True trajectory')
plt.plot(shaky_ex_smoothing_300, color='red', linewidth=1, alpha=1, label='Exponential smoothing')
plt.title('\n Fig.5. Calm Walk:True Trajectory, Exponential Smoothing - 300 points \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(-5, 305)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

Exponential smoothing showed a delay (the tracker is lagging behind panda). The running mean value has no delay with the true trajectory. Therefore, in this case, it is better to use Running mean, and Exponential smoothing requires changes to avoid delay like using forward-backward ES. 
ES uses previouse states and measurments due to previouse state we get delay between true trajectory amd smoothed by ES. RM uses only measurments by (window-1)/2 time in the past, present and in the future. So we didn't get delay in this case. 

#### Conclusion

During this assignment, we:
- generated a panda walking trajectory using random walk model with random noise which simulates random panda's motion (for calm walking with less noise variance and for "shaky cameras" greater noise variance)
- generate measurements of the panda’s trajectory with random noise which simulates noises of cameras and sensors (for calm walking with less noise variance and for "shaky cameras" greater noise variance)

- for calm walking:
	- using formulas for residual v[i], p[i] and math. expectations found estimated sigmas. The estimated values were very close to the true values we used for generation. This confirms that our model and identification method are correct. We saw that using more data points (3000) gives more accurate estimates than using fewer points (300)
	- found the optimal smoothing coefficient (α) for exponential smoothing. The value for 3000 points was more accurate.
	- performed exponential smoothing using the determined smoothing coefficient.The result is close to the true trajectory, proving the effectiveness of this method under calm conditions.

- for shaky cameras:
	- found optimal smoothing coefficient(0.25). This value is smaller than in the calm scenario because we need smaller alpha to filtrate greater noise.  
	- determined the window size (M = 7) for mean running using the determined alpha
	- made the visual comparison of the results. Exponential smoothing's delay is caused by its reliance on all past data. To fix this, a more advanced version like forward-backward exponential smoothing could be used. The running mean, by using a window size data, avoided this problem.

Quasi-optimal methods are valuable when the process model is unknown(like panda's walk), as they do not require knowledge of a model. However, a drawback is unknown estimatimation the error, which can lead to incorrect conclusions and requires ways to assess accuracy.
